# Cross the line — GPU steps (restyle + SAM 3 masks)

> **UNTESTED.** This notebook was written without GPU or Hugging Face access. Calls follow each model's
> model card / docs as of late 2025 and may need small fixes on first run (pipeline class names,
> argument names). Everything it produces is plain PNGs, so any other tool (a web UI, ComfyUI, an API)
> can substitute for any cell.

**What it makes (the contract with the CPU pipeline):**

| output | format | used by |
|---|---|---|
| `restyled/<style>_<model>_s<seed>.png` | same size as `clean_bg.png` (resized back if a model changes size) | `python -m crossover.run --restyled ...` |
| `masks/00000.png, 00001.png, …` | 8-bit, 255 = person, frame *i* = *i*-th frame decoded from the clip by OpenCV | `python -m crossover.run --masks masks/` |

Runtime: an A100 (40/80 GB) or L4 with CPU offload. Qwen-Image-Edit is ~20B parameters; FLUX.2 Klein 9B and SDXL fit more easily.

In [ ]:
# Setup. You need a Hugging Face token with access to the gated repos (FLUX.2 Klein, SAM 3).
!pip -q install -U diffusers transformers accelerate safetensors opencv-contrib-python-headless sentencepiece
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Upload / mount: the clip and the clean plate produced by `python -m crossover.run` (out/<run>/clean_bg.png).
# Easiest on Colab: put them in Google Drive.
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/crossover'          # <- change
VIDEO = f'{WORK}/sather_gate.mp4'                  # <- change
CLEAN = f'{WORK}/clean_bg.png'                     # <- from the CPU pipeline
import os; os.makedirs(f'{WORK}/restyled', exist_ok=True); os.makedirs(f'{WORK}/masks', exist_ok=True)

In [ ]:
import cv2, numpy as np, torch
from PIL import Image
clean_bgr = cv2.imread(CLEAN); H, W = clean_bgr.shape[:2]
clean_pil = Image.fromarray(cv2.cvtColor(clean_bgr, cv2.COLOR_BGR2RGB))

def model_size(w, h, max_side=1536, mult=16):
    # editors work best near ~1-2 MP; keep aspect, multiple of 16
    s = min(1.0, max_side / max(w, h))
    return int(w * s) // mult * mult, int(h * s) // mult * mult

def save_restyled(img_pil, name):
    out = cv2.cvtColor(np.array(img_pil.convert('RGB')), cv2.COLOR_RGB2BGR)
    out = cv2.resize(out, (W, H), interpolation=cv2.INTER_LANCZOS4)   # back to clip resolution
    path = f'{WORK}/restyled/{name}.png'; cv2.imwrite(path, out); print('wrote', path); return path

# Prompts: ask for the SAME scene - geometry must stay put (alignment only fixes small drift).
KEEP = ' Keep the exact same camera viewpoint, composition, buildings, and objects in the same places.'
PROMPTS = {
    'night':    'The same scene at night under moonlight, warm street lamps glowing, dark blue sky.' + KEEP,
    'century':  'The same place photographed around 1920: sepia photograph, period details, film grain.' + KEEP,
    'painting': 'The same scene as an impressionist oil painting with visible brush strokes.' + KEEP,
    'winter':   'The same scene in winter after heavy snowfall, snow on roofs and ground, overcast sky.' + KEEP,
}
SEEDS = [0, 1]

## A. Qwen-Image-Edit-2511 (instruction editing)

In [ ]:
from diffusers import DiffusionPipeline
qwen = DiffusionPipeline.from_pretrained('Qwen/Qwen-Image-Edit-2511', torch_dtype=torch.bfloat16)
qwen.enable_model_cpu_offload()          # remove on an 80 GB GPU for speed
print(type(qwen).__name__)               # expected: QwenImageEditPlusPipeline
mw, mh = model_size(W, H)
for style, prompt in PROMPTS.items():
    for seed in SEEDS:
        img = qwen(image=[clean_pil.resize((mw, mh))], prompt=prompt, negative_prompt=' ',
                   num_inference_steps=40, true_cfg_scale=4.0,
                   generator=torch.Generator('cuda').manual_seed(seed)).images[0]
        save_restyled(img, f'{style}_qwen_s{seed}')
del qwen; torch.cuda.empty_cache()

## B. FLUX.2 Klein 9B (reference-image editing)
`DiffusionPipeline` resolves the right pipeline class from the repo, so we don't hard-code it.

In [ ]:
from diffusers import DiffusionPipeline
flux = DiffusionPipeline.from_pretrained('black-forest-labs/FLUX.2-klein-9B', torch_dtype=torch.bfloat16)
flux.enable_model_cpu_offload()
print(type(flux).__name__)
mw, mh = model_size(W, H)
for style, prompt in PROMPTS.items():
    for seed in SEEDS:
        img = flux(image=clean_pil.resize((mw, mh)), prompt=prompt, width=mw, height=mh,
                   num_inference_steps=28, generator=torch.Generator('cuda').manual_seed(seed)).images[0]
        save_restyled(img, f'{style}_flux2klein_s{seed}')
del flux; torch.cuda.empty_cache()

## C. SDXL img2img + ControlNet (Canny edges) — the most geometry-preserving of the three

In [ ]:
from diffusers import ControlNetModel, StableDiffusionXLControlNetImg2ImgPipeline, AutoencoderKL
cn = ControlNetModel.from_pretrained('diffusers/controlnet-canny-sdxl-1.0', torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
sdxl = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', controlnet=cn, vae=vae, torch_dtype=torch.float16).to('cuda')
mw, mh = model_size(W, H, max_side=1344, mult=8)
src = clean_pil.resize((mw, mh))
edges = cv2.Canny(cv2.cvtColor(np.array(src), cv2.COLOR_RGB2GRAY), 80, 160)
control = Image.fromarray(np.stack([edges] * 3, -1))
for style, prompt in PROMPTS.items():
    for seed in SEEDS:
        img = sdxl(prompt=prompt.replace(KEEP, ''), image=src, control_image=control, strength=0.7,
                   controlnet_conditioning_scale=0.8, num_inference_steps=30, guidance_scale=6.0,
                   generator=torch.Generator('cuda').manual_seed(seed)).images[0]
        save_restyled(img, f'{style}_sdxlcanny_s{seed}')
del sdxl; torch.cuda.empty_cache()

## D. Pick the best editor per style by the seam metric
Copy the `crossover/` package folder into `WORK` (or `pip install` it) to score candidates right here.
Locally the same thing is: `python -m crossover.score --clean clean_bg.png --restyled restyled/*.png`

In [ ]:
import sys, glob; sys.path.insert(0, WORK)
from crossover.score import main as score_main
for style in PROMPTS:
    print('\n###', style)
    score_main(['--clean', CLEAN, '--restyled', *sorted(glob.glob(f'{WORK}/restyled/{style}_*.png'))])

## E. SAM 3: track every person from the text prompt "person"
Uses the 🤗 Transformers SAM 3 video API. If the class names differ in your transformers version,
the official repo (github.com/facebookresearch/sam3, `build_sam3_video_predictor`) does the same.

Frames are decoded with OpenCV exactly like the CPU pipeline, so mask *i* lines up with frame *i*.
For 4K clips set `SAM_MAX_SIDE` (masks are resized back up by the pipeline) and process in chunks.

In [ ]:
from transformers import Sam3VideoModel, Sam3VideoProcessor
SAM_MAX_SIDE = 1280
CHUNK = 300                      # frames per tracking session (memory); people re-detected per chunk
device = 'cuda'
model = Sam3VideoModel.from_pretrained('facebook/sam3').to(device, dtype=torch.bfloat16)
processor = Sam3VideoProcessor.from_pretrained('facebook/sam3')

def read_chunks(path, chunk, max_side):
    cap = cv2.VideoCapture(path); buf = []; start = 0
    while True:
        ok, f = cap.read()
        if not ok: break
        s = min(1.0, max_side / max(f.shape[:2]))
        if s < 1: f = cv2.resize(f, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        buf.append(Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
        if len(buf) == chunk: yield start, buf; start += len(buf); buf = []
    if buf: yield start, buf

n_written = 0
for start, frames in read_chunks(VIDEO, CHUNK, SAM_MAX_SIDE):
    session = processor.init_video_session(video=frames, inference_device=device,
                                           processing_device='cpu', video_storage_device='cpu', dtype=torch.bfloat16)
    session = processor.add_text_prompt(inference_session=session, text='person')
    done = set()
    for out in model.propagate_in_video_iterator(inference_session=session, max_frame_num_to_track=len(frames)):
        res = processor.postprocess_outputs(session, out)
        m = res['masks']                                           # (num_objects, h, w) bool/float
        union = (m.any(0) if m.dtype == torch.bool else (m > 0.5).any(0)) if len(m) else torch.zeros(frames[0].size[::-1], dtype=torch.bool)
        cv2.imwrite(f'{WORK}/masks/{start + out.frame_idx:05d}.png', union.cpu().numpy().astype(np.uint8) * 255)
        done.add(out.frame_idx); n_written += 1
    for j in set(range(len(frames))) - done:                         # frames with nobody
        cv2.imwrite(f'{WORK}/masks/{start + j:05d}.png', np.zeros(frames[0].size[::-1], np.uint8))
print('masks written:', n_written)

## F. Back on any machine (CPU is fine)
```bash
python -m crossover.run --video sather_gate.mp4 --scale 0.5 \
    --restyled restyled/night_sdxlcanny_s0.png --masks masks/ --local-warp --out out/sather_night
```
`--masks` also makes the clean plate ignore people who linger (masked median).